# Holt-Winters Method (Triple Exponential Smoothing)

Holt-Winters extends Holt's method by adding a **seasonal component**,
making it capable of forecasting data with both trend and seasonality.
This is the method most people mean when they say "exponential smoothing"
in practice.

Two variants exist:
- **Additive** — seasonal fluctuations are roughly constant in size.
- **Multiplicative** — seasonal fluctuations grow proportionally with the level.

**Key references:**
- FPP3 Section 8.3
- Portilla TSA Section 05: Triple Exponential Smoothing / Holt-Winters

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

## 1. Load Data

We use three datasets with trend + seasonality:
- **Airline passengers** — classic, multiplicative seasonality
- **Energy production** — long series with clear seasonal pattern
- **Alcohol sales** — multiplicative seasonality with strong December spikes

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

print(f"Airline passengers: {airline.shape}")
airline.head()

In [ ]:
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    index_col="DATE",
    parse_dates=True,
)
energy.index.freq = "MS"
energy.columns = ["Energy"]

print(f"Energy production: {energy.shape}")
energy.head()

In [ ]:
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]

print(f"Alcohol sales: {alcohol.shape}")
alcohol.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(airline["Passengers"])
axes[0].set_title("Airline Passengers")
axes[0].set_ylabel("Thousands")

axes[1].plot(energy["Energy"])
axes[1].set_title("Energy Production Index")
axes[1].set_ylabel("Index")

axes[2].plot(alcohol["Sales"])
axes[2].set_title("Alcohol Sales")
axes[2].set_ylabel("Millions $")

plt.tight_layout()
plt.show()

print("All three datasets show upward trend and seasonal patterns.")
print("Note how the seasonal amplitude grows with the level — multiplicative seasonality.")

## 2. Train / Test Split

In [ ]:
# Airline: 120 train, 24 test
air_train = airline.iloc[:120]
air_test = airline.iloc[120:]

# Energy: last 24 months for test
en_train = energy.iloc[:-24]
en_test = energy.iloc[-24:]

# Alcohol: last 24 months for test
alc_train = alcohol.iloc[:-24]
alc_test = alcohol.iloc[-24:]

print(f"Airline  — Train: {len(air_train)}, Test: {len(air_test)}")
print(f"Energy   — Train: {len(en_train)}, Test: {len(en_test)}")
print(f"Alcohol  — Train: {len(alc_train)}, Test: {len(alc_test)}")

---
## 3. Additive Holt-Winters — Component Form

When the seasonal pattern has **constant amplitude** (same size peaks and
troughs regardless of the level), use the **additive** variant:

$$
\text{Forecast:} \quad \hat{y}_{t+h|t} = \ell_t + h b_t + s_{t+h-m(k+1)}
$$

$$
\text{Level:} \quad \ell_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1} + b_{t-1})
$$

$$
\text{Trend:} \quad b_t = \beta^*(\ell_t - \ell_{t-1}) + (1-\beta^*) b_{t-1}
$$

$$
\text{Seasonal:} \quad s_t = \gamma(y_t - \ell_{t-1} - b_{t-1}) + (1-\gamma) s_{t-m}
$$

where $m$ is the seasonal period (12 for monthly data), $k$ is the integer
part of $(h-1)/m$, and we have **three smoothing parameters**:
- $\alpha$ — level smoothing
- $\beta^*$ — trend smoothing
- $\gamma$ — seasonal smoothing

The seasonal component $s_t$ is **added** to the level+trend, so the
seasonal indices sum to approximately zero over a full cycle.

In [ ]:
# Fit additive Holt-Winters on airline passengers
hw_add = ExponentialSmoothing(
    air_train["Passengers"],
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

print("Additive Holt-Winters parameters:")
print(f"  Alpha (level):    {hw_add.params['smoothing_level']:.4f}")
print(f"  Beta* (trend):    {hw_add.params['smoothing_trend']:.4f}")
print(f"  Gamma (seasonal): {hw_add.params['smoothing_seasonal']:.4f}")
print(f"  AIC:              {hw_add.aic:.2f}")

In [ ]:
hw_add.summary()

In [ ]:
fc_add = hw_add.forecast(len(air_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_add, label="HW Additive", color="tab:red", linestyle="--", linewidth=2)
ax.set_title("Additive Holt-Winters — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

mae_add = mean_absolute_error(air_test["Passengers"], fc_add)
print(f"MAE (Additive HW): {mae_add:.2f}")

---
## 4. Multiplicative Holt-Winters — Component Form

When the seasonal amplitude **grows proportionally with the level** (bigger
peaks at higher levels), use the **multiplicative** variant:

$$
\text{Forecast:} \quad \hat{y}_{t+h|t} = (\ell_t + h b_t) \cdot s_{t+h-m(k+1)}
$$

$$
\text{Level:} \quad \ell_t = \alpha \frac{y_t}{s_{t-m}} + (1-\alpha)(\ell_{t-1} + b_{t-1})
$$

$$
\text{Trend:} \quad b_t = \beta^*(\ell_t - \ell_{t-1}) + (1-\beta^*) b_{t-1}
$$

$$
\text{Seasonal:} \quad s_t = \gamma \frac{y_t}{\ell_{t-1} + b_{t-1}} + (1-\gamma) s_{t-m}
$$

The seasonal component $s_t$ is now a **multiplier** (values around 1.0),
and the seasonal indices average to approximately 1.0 over a full cycle.

In [ ]:
# Fit multiplicative Holt-Winters on airline passengers
hw_mul = ExponentialSmoothing(
    air_train["Passengers"],
    trend="mul",
    seasonal="mul",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

print("Multiplicative Holt-Winters parameters:")
print(f"  Alpha (level):    {hw_mul.params['smoothing_level']:.4f}")
print(f"  Beta* (trend):    {hw_mul.params['smoothing_trend']:.4f}")
print(f"  Gamma (seasonal): {hw_mul.params['smoothing_seasonal']:.4f}")
print(f"  AIC:              {hw_mul.aic:.2f}")

In [ ]:
hw_mul.summary()

In [ ]:
fc_mul = hw_mul.forecast(len(air_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_mul, label="HW Multiplicative", color="tab:green", linestyle="--", linewidth=2)
ax.set_title("Multiplicative Holt-Winters — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

mae_mul = mean_absolute_error(air_test["Passengers"], fc_mul)
print(f"MAE (Multiplicative HW): {mae_mul:.2f}")

---
## 5. Additive vs Multiplicative — Side by Side

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_add, label=f"Additive (MAE={mae_add:.1f})", color="tab:red",
        linestyle="--", linewidth=2)
ax.plot(fc_mul, label=f"Multiplicative (MAE={mae_mul:.1f})", color="tab:green",
        linestyle="--", linewidth=2)
ax.set_title("Additive vs Multiplicative Holt-Winters — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Additive AIC:       {hw_add.aic:.2f}")
print(f"Multiplicative AIC: {hw_mul.aic:.2f}")
print(f"\nFor airline passengers, multiplicative is more appropriate because")
print(f"the seasonal amplitude grows with the level.")

---
## 6. Visualizing the Additive vs Multiplicative Difference

The key difference between additive and multiplicative seasonality:

| | Additive | Multiplicative |
|---|---|---|
| Seasonal effect | Added to level+trend | Multiplied by level+trend |
| Seasonal indices | Sum to ~0 | Average to ~1.0 |
| Seasonal amplitude | Constant | Proportional to level |
| Best for | Constant seasonal swings | Growing seasonal swings |

In [ ]:
# Show the seasonal indices
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Additive seasonal indices
add_seasonal = hw_add.params['initial_seasons']
axes[0].bar(months, add_seasonal, color="tab:red", alpha=0.7)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_title("Additive Seasonal Indices")
axes[0].set_ylabel("Seasonal effect (added)")
axes[0].tick_params(axis="x", rotation=45)

# Multiplicative seasonal indices
mul_seasonal = hw_mul.params['initial_seasons']
axes[1].bar(months, mul_seasonal, color="tab:green", alpha=0.7)
axes[1].axhline(1.0, color="black", linestyle="--")
axes[1].set_title("Multiplicative Seasonal Indices")
axes[1].set_ylabel("Seasonal effect (multiplier)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Additive indices sum:       {sum(add_seasonal):.4f} (should be near 0)")
print(f"Multiplicative indices avg: {np.mean(mul_seasonal):.4f} (should be near 1)")

---
## 7. Decomposing the Holt-Winters Components

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Original data
axes[0].plot(air_train["Passengers"], color="tab:blue")
axes[0].set_title("Original Data")
axes[0].set_ylabel("Passengers")

# Level
axes[1].plot(hw_mul.level, color="tab:red")
axes[1].set_title("Level Component")
axes[1].set_ylabel("Level")

# Trend
axes[2].plot(hw_mul.trend, color="tab:green")
axes[2].set_title("Trend Component")
axes[2].set_ylabel("Trend")

# Seasonal
axes[3].plot(hw_mul.season, color="tab:purple")
axes[3].axhline(1.0, color="black", linestyle="--", alpha=0.5)
axes[3].set_title("Seasonal Component (multiplicative — values around 1.0)")
axes[3].set_ylabel("Seasonal")

plt.tight_layout()
plt.show()

---
## 8. Fitted Values

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Actual", color="tab:blue", alpha=0.7)
ax.plot(hw_mul.fittedvalues, label="Fitted (multiplicative HW)",
        color="tab:red", linestyle="--", alpha=0.8)
ax.set_title("Multiplicative Holt-Winters — Fitted Values")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

resid = air_train["Passengers"] - hw_mul.fittedvalues
print(f"Mean residual: {resid.mean():.4f}")
print(f"Std residual:  {resid.std():.4f}")

---
## 9. Damped Multiplicative Holt-Winters

We can also add damping to the Holt-Winters trend, combining the benefits
of damped trend with seasonal decomposition.

In [ ]:
hw_mul_damped = ExponentialSmoothing(
    air_train["Passengers"],
    trend="mul",
    seasonal="mul",
    seasonal_periods=12,
    damped_trend=True,
    initialization_method="estimated",
).fit(optimized=True)

fc_mul_damped = hw_mul_damped.forecast(len(air_test))

print("Damped Multiplicative Holt-Winters:")
print(f"  Alpha:  {hw_mul_damped.params['smoothing_level']:.4f}")
print(f"  Beta*:  {hw_mul_damped.params['smoothing_trend']:.4f}")
print(f"  Gamma:  {hw_mul_damped.params['smoothing_seasonal']:.4f}")
print(f"  Phi:    {hw_mul_damped.params['damping_trend']:.4f}")
print(f"  AIC:    {hw_mul_damped.aic:.2f}")

In [ ]:
mae_mul_damped = mean_absolute_error(air_test["Passengers"], fc_mul_damped)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_mul, label=f"Mul (MAE={mae_mul:.1f})", color="tab:green", linestyle="--")
ax.plot(fc_mul_damped, label=f"Mul Damped (MAE={mae_mul_damped:.1f})",
        color="tab:orange", linestyle="--")
ax.set_title("Multiplicative vs Damped Multiplicative Holt-Winters")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. All Trend/Seasonal Combinations

In [ ]:
configs = [
    {"trend": "add", "seasonal": "add", "damped": False, "label": "Add/Add"},
    {"trend": "add", "seasonal": "mul", "damped": False, "label": "Add/Mul"},
    {"trend": "mul", "seasonal": "add", "damped": False, "label": "Mul/Add"},
    {"trend": "mul", "seasonal": "mul", "damped": False, "label": "Mul/Mul"},
    {"trend": "add", "seasonal": "add", "damped": True, "label": "Add/Add (damped)"},
    {"trend": "mul", "seasonal": "mul", "damped": True, "label": "Mul/Mul (damped)"},
]

results = []
for cfg in configs:
    try:
        m = ExponentialSmoothing(
            air_train["Passengers"],
            trend=cfg["trend"],
            seasonal=cfg["seasonal"],
            seasonal_periods=12,
            damped_trend=cfg["damped"],
            initialization_method="estimated",
        ).fit(optimized=True)
        fc = m.forecast(len(air_test))
        results.append({
            "Model": cfg["label"],
            "AIC": m.aic,
            "MAE": mean_absolute_error(air_test["Passengers"], fc),
        })
    except Exception as e:
        results.append({"Model": cfg["label"], "AIC": np.nan, "MAE": np.nan})

results_df = pd.DataFrame(results).round(2)
results_df = results_df.sort_values("AIC").reset_index(drop=True)
print(results_df.to_string(index=False))

---
## 11. Energy Production — Another Example

In [ ]:
hw_energy = ExponentialSmoothing(
    en_train["Energy"],
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

fc_energy = hw_energy.forecast(len(en_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(en_train["Energy"].iloc[-120:], label="Train (last 10 yr)", color="black", alpha=0.5)
ax.plot(en_test["Energy"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_energy, label="HW Additive", color="tab:red", linestyle="--", linewidth=2)
ax.set_title("Holt-Winters Additive — Energy Production")
ax.set_ylabel("Energy Index")
ax.legend()
plt.tight_layout()
plt.show()

print(f"MAE: {mean_absolute_error(en_test['Energy'], fc_energy):.2f}")

---
## 12. Alcohol Sales — Multiplicative Seasonality

In [ ]:
hw_alc = ExponentialSmoothing(
    alc_train["Sales"],
    trend="mul",
    seasonal="mul",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

fc_alc = hw_alc.forecast(len(alc_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(alc_train["Sales"].iloc[-120:], label="Train (last 10 yr)", color="black", alpha=0.5)
ax.plot(alc_test["Sales"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_alc, label="HW Multiplicative", color="tab:green", linestyle="--", linewidth=2)
ax.set_title("Holt-Winters Multiplicative — Alcohol Sales")
ax.set_ylabel("Sales (millions $)")
ax.legend()
plt.tight_layout()
plt.show()

mae_alc = mean_absolute_error(alc_test["Sales"], fc_alc)
print(f"MAE: {mae_alc:.2f}")
print(f"\nAlpha: {hw_alc.params['smoothing_level']:.4f}")
print(f"Beta*: {hw_alc.params['smoothing_trend']:.4f}")
print(f"Gamma: {hw_alc.params['smoothing_seasonal']:.4f}")

---
## 13. Prediction Intervals

Holt-Winters can provide **prediction intervals** that quantify forecast
uncertainty.  The intervals widen as the forecast horizon increases,
reflecting growing uncertainty about the future.

In [ ]:
# Use the simulate method for prediction intervals
# statsmodels ExponentialSmoothing provides prediction intervals via simulation
hw_mul_refit = ExponentialSmoothing(
    air_train["Passengers"],
    trend="mul",
    seasonal="mul",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

# Generate simulations for prediction intervals
n_sim = 1000
simulations = hw_mul_refit.simulate(
    nsimulations=len(air_test),
    repetitions=n_sim,
    anchor="end",
)

# Compute percentiles
fc_mean_sim = simulations.mean(axis=1)
fc_lower_80 = simulations.quantile(0.10, axis=1)
fc_upper_80 = simulations.quantile(0.90, axis=1)
fc_lower_95 = simulations.quantile(0.025, axis=1)
fc_upper_95 = simulations.quantile(0.975, axis=1)

print(f"Generated {n_sim} simulations for prediction intervals.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(air_train["Passengers"].iloc[-48:], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)

# Point forecast
fc_point = hw_mul_refit.forecast(len(air_test))
ax.plot(fc_point, label="Forecast", color="tab:red", linewidth=2)

# Prediction intervals
ax.fill_between(fc_lower_95.index, fc_lower_95, fc_upper_95,
                color="tab:red", alpha=0.15, label="95% PI")
ax.fill_between(fc_lower_80.index, fc_lower_80, fc_upper_80,
                color="tab:red", alpha=0.25, label="80% PI")

ax.set_title("Multiplicative Holt-Winters with Prediction Intervals")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("Prediction intervals widen with horizon — we are less certain")
print("about forecasts further into the future.")

---
## 14. Choosing Additive vs Multiplicative

A practical decision rule:

1. **Plot the data**: if the seasonal amplitude grows with the level,
   use multiplicative. If it stays constant, use additive.
2. **Try both**: fit both variants, compare AIC or out-of-sample MAE.
3. **Log transform trick**: applying `log()` to the data and then using
   additive seasonality is approximately equivalent to multiplicative
   seasonality on the original scale.

In [ ]:
# Demonstrate: the seasonal amplitude grows with level for airline data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# First few years
axes[0].plot(airline["Passengers"].iloc[:36])
axes[0].set_title("Early years (1949-1951): small seasonal swings")
axes[0].set_ylabel("Passengers")
y_range_early = airline["Passengers"].iloc[:36].max() - airline["Passengers"].iloc[:36].min()

# Last few years
axes[1].plot(airline["Passengers"].iloc[-36:])
axes[1].set_title("Late years (1958-1960): large seasonal swings")
axes[1].set_ylabel("Passengers")
y_range_late = airline["Passengers"].iloc[-36:].max() - airline["Passengers"].iloc[-36:].min()

# Same y-axis scale for comparison
for a in axes:
    a.set_ylim(80, 650)

plt.tight_layout()
plt.show()

print(f"Range in early years: {y_range_early}")
print(f"Range in late years:  {y_range_late}")
print(f"The seasonal amplitude clearly grows — multiplicative is appropriate.")

---
## 15. Full Comparison Table — All Methods So Far

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt

# SES
ses_fit = SimpleExpSmoothing(
    air_train["Passengers"], initialization_method="estimated"
).fit(optimized=True)
fc_ses = ses_fit.forecast(len(air_test))

# Holt linear
holt_fit = Holt(
    air_train["Passengers"], initialization_method="estimated"
).fit(optimized=True)
fc_holt = holt_fit.forecast(len(air_test))

# Holt damped
holt_d_fit = Holt(
    air_train["Passengers"], damped_trend=True, initialization_method="estimated"
).fit(optimized=True)
fc_holt_d = holt_d_fit.forecast(len(air_test))

actual = air_test["Passengers"].values

all_results = pd.DataFrame([
    {"Method": "SES", "AIC": ses_fit.aic,
     "MAE": mean_absolute_error(actual, fc_ses)},
    {"Method": "Holt Linear", "AIC": holt_fit.aic,
     "MAE": mean_absolute_error(actual, fc_holt)},
    {"Method": "Holt Damped", "AIC": holt_d_fit.aic,
     "MAE": mean_absolute_error(actual, fc_holt_d)},
    {"Method": "HW Additive", "AIC": hw_add.aic,
     "MAE": mean_absolute_error(actual, fc_add)},
    {"Method": "HW Multiplicative", "AIC": hw_mul.aic,
     "MAE": mean_absolute_error(actual, fc_mul)},
    {"Method": "HW Mul Damped", "AIC": hw_mul_damped.aic,
     "MAE": mean_absolute_error(actual, fc_mul_damped)},
]).round(2)

all_results = all_results.sort_values("MAE").reset_index(drop=True)
print(all_results.to_string(index=False))
print(f"\nBest method: {all_results.iloc[0]['Method']}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(air_train["Passengers"].iloc[-36:], color="black", alpha=0.4, label="Train")
ax.plot(air_test["Passengers"], color="tab:blue", linewidth=2, label="Actual")
ax.plot(fc_ses, linestyle="--", label="SES", alpha=0.7)
ax.plot(fc_holt, linestyle="--", label="Holt", alpha=0.7)
ax.plot(fc_add, linestyle="--", label="HW Add", alpha=0.7)
ax.plot(fc_mul, linestyle="--", linewidth=2, label="HW Mul")
ax.set_title("All Exponential Smoothing Methods — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | Detail |
|---------|--------|
| **What** | SES + trend + seasonal component |
| **Parameters** | $\alpha$ (level), $\beta^*$ (trend), $\gamma$ (seasonal) |
| **Additive** | Constant seasonal amplitude; seasonal indices sum to 0 |
| **Multiplicative** | Growing seasonal amplitude; seasonal indices average to 1 |
| **Damped variants** | Add $\phi$ to prevent indefinite trend extrapolation |
| **Choosing add vs mul** | Look at data: growing amplitude = multiplicative |
| **Prediction intervals** | Available via simulation; widen with horizon |
| **statsmodels** | `ExponentialSmoothing(trend=, seasonal=, seasonal_periods=)` |

**Next notebook**: The ETS framework — a systematic taxonomy of all
exponential smoothing models with automatic selection via AIC.

In [ ]:
print("Next notebook: ETS Framework — the state-space formulation")
print("that unifies all exponential smoothing methods under one roof.")